# ShopSmart - Data Preprocessing

## 1. Product Text Preparation

We prepare product metadata for content-based recommendation.

The objective is to combine product titles, categories,
features, and descriptions into a single textual representation.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

ratings = pd.read_csv(
    PROCESSED_DIR / "ratings_shopsmart.csv"
)

products = pd.read_json(
    PROCESSED_DIR / "products_shopsmart.jsonl",
    lines=True
)

print("Ratings:", ratings.shape)
print("Products:", products.shape)

display(products.head(3))

Ratings: (147572, 4)
Products: (2000, 8)


,parent_asin,title,main_category,categories,features,description,price,store
0,B07RWDZ3HG,SilverStone Technology SST-FF121 120mm Ultra F...,All Electronics,"[Electronics, Computers & Accessories, Compute...","[Honeycomb grill increases airflow focus, Inte...",[Finding a quiet and efficient cooling fan is ...,6.47,SilverStone Technology
1,B09SJ9192T,"SAMSUNG 23.5” CF396 Curved Computer Monitor, A...",Computers,"[Electronics, Computers & Accessories, Monitors]",[IMMERSIVE VIEWING EXPERIENCE: The 1800R curva...,[Enjoy a truly immersive viewing experience wi...,158.00,SAMSUNG
2,B07FTSFGXH,"Cat6 Ethernet Cable, 12 Feet (2 Pack) LAN, utp...",Industrial & Scientific,"[Electronics, Computers & Accessories, Compute...",[QUALITY CONTROL - Each Cat 6 internet cable 5...,[],11.99,Ultra Clarity Cables


In [2]:
def list_to_text(value):
    if isinstance(value, list):
        return " ".join(
            str(item) for item in value if item is not None
        )
    return ""


products["title"] = products["title"].fillna("")
products["main_category"] = products["main_category"].fillna("")

for col in ["categories", "features", "description"]:
    products[col + "_text"] = products[col].apply(list_to_text)


products["content"] = (
    products["title"] + " " +
    products["main_category"] + " " +
    products["categories_text"] + " " +
    products["features_text"] + " " +
    products["description_text"]
).str.strip()

display(
    products[["parent_asin", "title", "content"]].head()
)

,parent_asin,title,content
0,B07RWDZ3HG,SilverStone Technology SST-FF121 120mm Ultra F...,SilverStone Technology SST-FF121 120mm Ultra F...
1,B09SJ9192T,"SAMSUNG 23.5” CF396 Curved Computer Monitor, A...","SAMSUNG 23.5” CF396 Curved Computer Monitor, A..."
2,B07FTSFGXH,"Cat6 Ethernet Cable, 12 Feet (2 Pack) LAN, utp...","Cat6 Ethernet Cable, 12 Feet (2 Pack) LAN, utp..."
3,B00RVFCFI2,ExpertPower XP-669 7.5-Inch Dual Band SMA-F An...,ExpertPower XP-669 7.5-Inch Dual Band SMA-F An...
4,B0C49KQRX4,Tzowla Travel Laptop Backpack Water Resistant ...,Tzowla Travel Laptop Backpack Water Resistant ...


In [3]:
products["content_length"] = products["content"].str.len()

print("Content length statistics:")
display(products["content_length"].describe())

print("\nProducts with empty content:")
print((products["content_length"] == 0).sum())

print("\nProducts with less than 50 characters:")
print((products["content_length"] < 50).sum())

print("\nExample product content:")
print(products["content"].iloc[0][:1000])

Content length statistics:


count     2000.000000
mean      1648.106000
std       1475.732592
min         54.000000
25%        920.000000
50%       1332.000000
75%       1905.750000
max      28312.000000
Name: content_length, dtype: float64


Products with empty content:
0

Products with less than 50 characters:
0

Example product content:
SilverStone Technology SST-FF121 120mm Ultra Fine Fan Filter with Magnet Cooling,black All Electronics Electronics Computers & Accessories Computer Components Internal Components Fans & Cooling Case Fans Honeycomb grill increases airflow focus Integrated filter provides class leading dust reduction Compatible with all standard 120mm fans Include 4 screws Finding a quiet and efficient cooling fan is the ultimate goal for every enthusiast. The SST-FF121 is a fan grille designed for easy installation with any 120mm fan. The honeycomb pattern on the grille not only reduces wind shear noise but also channels the airflow into more focused beam. Its integrated filter also provides reduced dust buildup to extend the life cycle of components. All in all, the SST-FF121 is an excellent upgrade for any 120mm fans in your system.


In [4]:
products.to_json(
    PROCESSED_DIR / "products_preprocessed.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Preprocessed product dataset saved!")

Preprocessed product dataset saved!


## 2. Chronological Train / Validation / Test Split

We split user interactions chronologically to simulate
real-world recommendation scenarios.

For each user:
- Last interaction: Test
- Second-to-last interaction: Validation
- Remaining interactions: Training

In [5]:
print("Missing timestamps:", ratings["timestamp"].isna().sum())

print(
    "Duplicate user-product pairs:",
    ratings.duplicated(
        subset=["user_id", "parent_asin"]
    ).sum()
)

# Sort interactions chronologically for each user
ratings_sorted = ratings.sort_values(
    by=["user_id", "timestamp", "parent_asin"]
).reset_index(drop=True)

display(ratings_sorted.head())

Missing timestamps: 0
Duplicate user-product pairs: 0


,user_id,parent_asin,rating,timestamp
0,AE22LPCN47WUTHSG67R6SKN4A4MQ,B00XWE00F6,4.0,1305319536000
1,AE22LPCN47WUTHSG67R6SKN4A4MQ,B0BV93KQZR,4.0,1336185172000
2,AE22LPCN47WUTHSG67R6SKN4A4MQ,B01LW9B75D,4.0,1336186041000
3,AE22LPCN47WUTHSG67R6SKN4A4MQ,B07C1263TQ,5.0,1339626349000
4,AE22LPCN47WUTHSG67R6SKN4A4MQ,B07TZP1SD3,5.0,1407023629000


In [9]:
# Position of each interaction within its user's history
ratings_sorted["rank"] = (
    ratings_sorted.groupby("user_id")
    .cumcount()
)

# Number of interactions for each user
ratings_sorted["total"] = (
    ratings_sorted.groupby("user_id")["user_id"]
    .transform("size")
)

# Training: all except last two interactions
train = ratings_sorted[
    ratings_sorted["rank"] < ratings_sorted["total"] - 2
].copy()

# Validation: second-to-last interaction
validation = ratings_sorted[
    ratings_sorted["rank"] == ratings_sorted["total"] - 2
].copy()

# Test: last interaction
test = ratings_sorted[
    ratings_sorted["rank"] == ratings_sorted["total"] - 1
].copy()

train = train.drop(columns=["rank", "total"])
validation = validation.drop(columns=["rank", "total"])
test = test.drop(columns=["rank", "total"])

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (132836, 4)
Validation: (7368, 4)
Test: (7368, 4)


In [10]:
print("Users in train:", train["user_id"].nunique())
print("Users in validation:", validation["user_id"].nunique())
print("Users in test:", test["user_id"].nunique())

print("\nProducts in train:", train["parent_asin"].nunique())
print("Products in validation:", validation["parent_asin"].nunique())
print("Products in test:", test["parent_asin"].nunique())

# Check that all interactions were preserved
assert len(train) + len(validation) + len(test) == len(ratings)

# Check that each user has sufficient training history
assert train.groupby("user_id").size().min() >= 13

# Check for cold-start products in validation and test
train_products = set(train["parent_asin"])

val_cold_items = set(validation["parent_asin"]) - train_products
test_cold_items = set(test["parent_asin"]) - train_products

print("\nCold-start products in validation:", len(val_cold_items))
print("Cold-start products in test:", len(test_cold_items))

Users in train: 7368
Users in validation: 7368
Users in test: 7368

Products in train: 1998
Products in validation: 1503
Products in test: 1483

Cold-start products in validation: 2
Cold-start products in test: 2


In [11]:
train.to_csv(
    PROCESSED_DIR / "train.csv",
    index=False
)

validation.to_csv(
    PROCESSED_DIR / "validation.csv",
    index=False
)

test.to_csv(
    PROCESSED_DIR / "test.csv",
    index=False
)

print("Train, validation and test datasets saved!")

Train, validation and test datasets saved!
